# Video RAG 파이프라인 (긴 영상)

18분짜리 1949년 우루과이 여행 다큐멘터리를 사용해, 3강에서 배운
**Multi-Vector Retriever 패턴**을 영상에 적용합니다.

- **Vector Store**: 클립 요약의 임베딩만 저장 (검색 전용)
- **Document Store**: 클립의 원본 프레임 (답변 생성 전용)

## 사전 준비
```bash
uv sync
ollama pull qwen3-vl:4b
ollama pull qwen3-embedding:0.6b
```

In [ ]:
import chromadb

import video_utils as vu

VIDEO = "data/4_long_video_uruguay_1949.mp4"
info = vu.get_video_info(VIDEO)
info

## 1. 클립 분할
18분 영상을 한 번에 VLM에 넣을 수 없으므로, 3분 단위(총 6개) 클립으로 나눕니다.
각 클립에서는 대표 프레임 3장만 뽑아 요약에 사용합니다 (요약은 검색 전용이라
프레임 수를 적게 써도 무방).

In [ ]:
CLIP_LEN_SEC = 180  # 3분

clips = []
start = 0.0
clip_id = 0
while start < info["duration_sec"]:
    end = min(start + CLIP_LEN_SEC, info["duration_sec"])
    clips.append({"clip_id": clip_id, "start": start, "end": end})
    start = end
    clip_id += 1

for c in clips:
    print(f"clip {c['clip_id']}: {c['start']:.0f}s ~ {c['end']:.0f}s")

## 2. 클립별 대표 프레임 추출 + 요약 생성 (Document Store + 요약)
각 클립 구간 안에서만 균등 샘플링으로 3장을 뽑고, VLM으로 짧은 요약을 만듭니다.
원본 프레임은 `documents` 딕셔너리(Document Store)에, 요약 텍스트는 뒤에서
임베딩해 Vector Store에 저장합니다.

In [ ]:
documents = {}  # clip_id -> frames (Document Store, 답변 생성 전용)

for c in clips:
    frames = vu.extract_frames_range(VIDEO, c["start"], c["end"], n_frames=3)
    documents[c["clip_id"]] = frames
    summary = vu.ask_video(
        frames,
        "아래는 여행 다큐멘터리의 한 구간에서 뽑은 프레임이야. 이 구간에서 "
        "어떤 장소/장면이 나오는지 한국어 한 문장으로 요약해줘.",
        with_timestamps=False,
    )
    c["summary"] = summary.strip()
    print(f"[clip {c['clip_id']}] {c['start']:.0f}~{c['end']:.0f}s: {c['summary']}")

## 3. 요약 임베딩 → Vector Store (Chroma)
3강과 동일한 패턴: 검색은 요약 텍스트의 임베딩으로 하고, 원본은 `clip_id`로
다시 찾아옵니다.

In [ ]:
client = chromadb.Client()
collection = client.create_collection("video_clips")

for c in clips:
    embedding = vu.embed_text(c["summary"])
    collection.add(
        ids=[str(c["clip_id"])],
        embeddings=[embedding],
        documents=[c["summary"]],
        metadatas=[{"start": c["start"], "end": c["end"]}],
    )

print("저장된 클립 수:", collection.count())

## 4. 질의 → 유사 클립 검색 → 원본 프레임으로 답변 생성

In [ ]:
def video_rag_query(question, top_k=1):
    q_embedding = vu.embed_text(question)
    result = collection.query(query_embeddings=[q_embedding], n_results=top_k)

    clip_id = int(result["ids"][0][0])
    meta = result["metadatas"][0][0]
    matched_summary = result["documents"][0][0]
    frames = documents[clip_id]

    print(f"검색된 구간: {meta['start']:.0f}s ~ {meta['end']:.0f}s")
    print(f"클립 요약: {matched_summary}\n")

    answer = vu.ask_video(
        frames,
        f"질문: {question}\n"
        f"위 프레임은 영상의 {meta['start']:.0f}초~{meta['end']:.0f}초 구간이야. "
        "이 구간을 근거로 답하고, 답변에 몇 분 몇 초 구간인지도 함께 알려줘. "
        "프레임에서 확인되지 않으면 '확인 불가'라고 답해.",
        with_timestamps=False,
    )
    vu.show_frames(frames)
    return answer


print(video_rag_query("이 영상에서 도시의 거리나 건물이 나오는 장면을 찾아줘"))

In [ ]:
print(video_rag_query("농촌이나 목장, 동물이 나오는 장면이 있어?"))

## 이미지 RAG와의 차이 확인
- 검색 결과가 "K번째 클립"이 아니라 **N초~N초 구간**으로 표시되도록 설계 →
  사용자가 실제 영상에서 바로 확인 가능
- 클립 길이(`CLIP_LEN_SEC`)를 줄이면 검색은 더 정밀해지지만 클립 수(=임베딩/저장
  비용, VLM 호출 횟수)가 늘어납니다 — 5강 이론의 트레이드오프 표와 동일한 원리